# Full review of the top 1,000 dataset — 5 September 2026

**Assessment: needs revision for a bands-only or city-completeness claim; usable with caveats as a frozen Spotify catalogue.** All 1,000 rows receive structural, provenance, cross-file and origin-status checks. Existing raw MusicBrainz records and 970 frozen Spotify biographies are screened across the catalogue; external artist/label sources verify selected eligibility conflicts. This is not a claim of 1,000 independent manual biographical adjudications.

The notebook reads frozen inputs and writes only its own evidence files under `artifacts/reviews/top1000_20260905`. It does not refresh metrics or change the catalogue. Run all cells from the repository environment. Source hashes preserve the precise reviewed state.

In [1]:
from pathlib import Path
import os, sys, json, gzip, re, hashlib, importlib.util
import pandas as pd
import numpy as np
from pandas.testing import assert_frame_equal
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/python_uk_bands').exists())
os.environ.setdefault('MPLCONFIGDIR', '/tmp/top1000-review-mpl')
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
from python_uk_bands.popularity_first import select_top_groups, attach_fua_population, build_population_adjusted_metrics, build_origin_concentration
from python_uk_bands.output_share import build_output_share_metrics
from python_uk_bands.scaling_models import fit_negative_binomial_band_scaling, fit_loglog_follower_scaling
from python_uk_bands.matching import normalize_name
from scripts.build_popularity_first_candidates import build_candidates
from scripts.audit_top1000_origins import build_audit

OUT = ROOT / 'artifacts/reviews/top1000_20260905'
OUT.mkdir(parents=True, exist_ok=True)
sources, checks = {}, []
def read(path):
    p = ROOT / path
    sources[path] = hashlib.sha256(p.read_bytes()).hexdigest()
    return pd.read_csv(p, keep_default_na=False) if p.suffix == '.csv' else json.loads(p.read_text())
def check(name, condition):
    assert condition, name
    checks.append({'check': name, 'result': 'passed'})
def compare(name, actual, expected):
    actual, expected = actual.copy(), expected.copy()
    for column in actual:
        if pd.api.types.is_numeric_dtype(actual[column]) and not pd.api.types.is_bool_dtype(actual[column]):
            expected[column] = pd.to_numeric(expected[column].replace('', np.nan))
        else:
            actual[column] = actual[column].fillna('')
            expected[column] = expected[column].fillna('')
    assert_frame_equal(actual, expected, check_dtype=False, rtol=1e-10, atol=1e-8)
    checks.append({'check': name, 'result': 'passed'})

base = 'popularity_first_top1000_20260718T204522Z'
c = read(f'data/processed/{base}_bands.csv')
candidates = read('data/interim/uk_group_spotify_candidates_20260718T201100Z.csv')
metrics = read('data/processed/uk_group_spotify_metrics_20260718T204522Z.csv')
identity = read(f'data/interim/{base}_identity_audit.csv')
overrides = read('reference/popularity_first_top1000_overrides_20260718.csv')
fua = read(f'data/interim/{base}_fua_mapping_audit.csv')
mapping = read('reference/popularity_first_top1000_origin_fua_mapping_20260718.csv')
population = read('data/processed/uk_fua_population_2024_20260830T221015Z.csv')
origins = read('data/processed/top1000_origin_fact_check_20260901.csv')
origin_report = read('data/processed/top1000_origin_fact_check_20260901_report.json')
decisions = read('reference/top1000_origin_fact_check_decisions_20260902.csv')
dashboard = read('interactive/public/data/dashboard.json')
wikidata = read('data/raw/wikidata/review_extension_entities_20260725.json')
musicbrainz = read('data/raw/musicbrainz/top1000_origin_areas_20260901.json')
wikipedia = read('data/raw/wikipedia/top1000_origin_infobox_20260901.json')
capture = read('data/raw/spotify/uk_group_spotify_metrics_20260718T204522Z_report.json')
raw_path = 'data/raw/spotify/uk_group_spotify_metrics_20260718T204522Z.json.gz'
sources[raw_path] = hashlib.sha256((ROOT / raw_path).read_bytes()).hexdigest()
with gzip.open(ROOT / raw_path, 'rt') as handle:
    raw = json.load(handle)
total = int(c.monthly_listeners.sum())
followers_total = int(c.followers.sum())
display(c[['popularity_rank','spotify_name','monthly_listeners','followers','origin_cluster']].head(10))

,popularity_rank,spotify_name,monthly_listeners,followers,origin_cluster
0,1,Coldplay,92034104,64732379,London
1,2,Arctic Monkeys,52301870,35028298,Sheffield
2,3,Queen,51601234,58004701,London
3,4,Fleetwood Mac,50959935,15546243,London
4,5,One Direction,45724043,41728665,London
5,6,Radiohead,43168156,16613975,Oxford
6,7,The Police,39887109,8458469,London
7,8,Oasis,39676473,13941515,Manchester
8,9,Gorillaz,38173160,16369649,London
9,10,The Beatles,37927636,32385954,Liverpool


## Keys, metrics and selection reproduce

The grain is one selected Spotify artist/entity, ranked by captured global monthly listeners. These are artist-level counts, not distinct people across the whole catalogue. The single timestamp is the capture batch timestamp, not proof of simultaneous measurement.

In [2]:
check('Exactly 1,000 rows and contiguous ordered ranks', len(c) == 1000 and c.popularity_rank.tolist() == list(range(1, 1001)))
for key in ['capture_key','returned_spotify_id','requested_spotify_id','wikidata_qid']:
    check(f'Unique nonblank {key}', c[key].is_unique and c[key].ne('').all())
check('Identifier formats', c.returned_spotify_id.str.fullmatch(r'[A-Za-z0-9]{22}').all() and c.wikidata_qid.str.fullmatch(r'Q\d+').all())
check('Nonnegative integer audience measures', c[['monthly_listeners','followers']].ge(0).all().all() and c[['monthly_listeners','followers']].mod(1).eq(0).all().all())
check('Descending popularity', c.monthly_listeners.is_monotonic_decreasing)
check('No exact duplicate rows', not c.duplicated().any())
raw_frame = read('data/raw/wikidata/uk_group_candidates_with_spotify_20260718T201100Z.json')
compare('Candidate frame rebuild', build_candidates(raw_frame), candidates)
selected, rebuilt_identity = select_top_groups(candidates, metrics, overrides, 1000)
compare('Selected catalogue rebuild', selected, c)
compare('Identity audit rebuild', rebuilt_identity, identity)
compare('Origin totals rebuild', build_origin_concentration(c), read(f'data/processed/{base}_origins.csv'))
check('Capture outcomes partition the candidate frame', len(metrics) + len(capture['metric_failures']) == len(candidates) and set(metrics.band) | {x['band'] for x in capture['metric_failures']} == set(candidates.capture_key))
raw_errors = []
for row in c.itertuples():
    artist = raw['artist_overviews'][row.capture_key]['data']['artist']
    expected = (row.returned_spotify_id, row.spotify_name, row.monthly_listeners, row.followers)
    actual = (artist['id'], artist['profile']['name'], artist['stats']['monthlyListeners'], artist['stats']['followers'])
    if actual != expected: raw_errors.append(row.spotify_name)
check('All selected IDs, names and audience counts match raw Spotify responses', not raw_errors)
pool = identity[identity.identity_accepted & identity.band_eligible & ~identity.redirect_duplicate & ~identity.entity_duplicate]
cutoff = int(c.monthly_listeners.min())
funnel = {'wikidata_bindings': len(raw_frame['results']['bindings']), 'candidate_ids': len(candidates), 'metric_rows': len(metrics), 'metric_failures': len(capture['metric_failures']), 'eligible_pool': len(pool), 'selected':len(c), 'cutoff':cutoff, 'next_eligible':int(pool.iloc[1000].monthly_listeners), 'cutoff_gap':cutoff-int(pool.iloc[1000].monthly_listeners), 'near_cutoff_10pct':int(pool.monthly_listeners.between(cutoff*.9,cutoff*1.1).sum())}
display(pd.Series(funnel))

wikidata_bindings     1928
candidate_ids         1775
metric_rows           1749
metric_failures         26
eligible_pool         1633
selected              1000
cutoff               29728
next_eligible        29347
cutoff_gap             381
near_cutoff_10pct       53
dtype: int64

## Eligibility needs review beyond Wikidata type labels

Five verified solo projects are retained by the current group-type rule. The row ledger records the entire MusicBrainz type screen and biography keyword screen as leads, not automatic exclusions: a band's biography can mention a member's solo career. Historical groups now operating as solo acts also need an explicit convention.

Primary evidence: [IAMX press kit](https://iamxmusic.com/pages/electronic-press-kit), [Ex:Re label page](https://shop.4ad.com/format/1134848-exre), [Russ Davies alter egos](https://www.russ-davies.com/alter-egos), [Hallucinogen album credits](https://hallucinogenmusic.bandcamp.com/album/twisted), and [Sidewalks and Skeletons artist biography](https://ra.co/dj/sidewalksandskeletons/biography). Frozen Spotify biographies independently support Sidewalks and Skeletons, Ex:Re, Cinnamon Chasers and Hallucinogen.

In [3]:
verified_solo = {
    'Sidewalks and Skeletons': 'https://ra.co/dj/sidewalksandskeletons/biography',
    'IAMX': 'https://iamxmusic.com/pages/electronic-press-kit',
    'Cinnamon Chasers': 'https://www.russ-davies.com/alter-egos',
    'Ex:Re': 'https://shop.4ad.com/format/1134848-exre',
    'Hallucinogen': 'https://hallucinogenmusic.bandcamp.com/album/twisted',
}
pattern = re.compile(r'solo[ -](?:project|artist|venture|vehicle|alias|act)|(?:alias|moniker|pseudonym|alter ego) of|one.man|one.person|brainchild of', re.I)
ledger = c[['popularity_rank','returned_spotify_id','wikidata_qid','spotify_name','monthly_listeners','followers','origin_cluster','origin_resolution']].copy()
bio_flags, bio_present, person_matches, member_links = [], [], [], []
mb_problems = []
for row in c.itertuples():
    bio = raw['artist_overviews'][row.capture_key]['data']['artist']['profile']['biography'].get('text') or ''
    bio_present.append(bool(bio)); bio_flags.append(bool(pattern.search(bio)))
    claims = wikidata['entities'].get(row.wikidata_qid, {}).get('claims', {}).get('P434', [])
    records = [musicbrainz['artists'].get(x.get('mainsnak',{}).get('datavalue',{}).get('value'), {}) for x in claims]
    persons = [x for x in records if x.get('type') == 'Person']
    matching = [x['name'] for x in persons if normalize_name(x.get('name','')) == normalize_name(row.spotify_name)]
    unrelated = [x['name'] for x in persons if normalize_name(x.get('name','')) != normalize_name(row.spotify_name)]
    person_matches.append('|'.join(matching)); member_links.append('|'.join(unrelated))
    if persons: mb_problems.append({'spotify_name':row.spotify_name, 'person_records':[x['name'] for x in persons]})
ledger['spotify_biography_present'] = bio_present
ledger['biography_keyword_review'] = bio_flags
ledger['matching_musicbrainz_person'] = person_matches
ledger['other_person_ids_in_musicbrainz_links'] = member_links
ledger['eligibility_review'] = ledger.spotify_name.map(lambda n: 'verified_solo_project' if n in verified_solo else '')
ledger['eligibility_source_url'] = ledger.spotify_name.map(verified_solo).fillna('')
solo = ledger[ledger.eligibility_review.ne('')]
check('All five documented solo-project conflicts exist in the reviewed catalogue', len(solo)==5)
display(solo[['popularity_rank','spotify_name','monthly_listeners','eligibility_source_url']])
print('MusicBrainz matching-person leads:', int(ledger.matching_musicbrainz_person.ne('').sum()))
print('Biographies available / keyword leads:', sum(bio_present), sum(bio_flags))
print('Group records also linked to individual members:', int(ledger.other_person_ids_in_musicbrainz_links.ne('').sum()))

,popularity_rank,spotify_name,monthly_listeners,eligibility_source_url
123,124,Sidewalks and Skeletons,4198181,https://ra.co/dj/sidewalksandskeletons/biography
331,332,IAMX,750077,https://iamxmusic.com/pages/electronic-press-kit
528,529,Cinnamon Chasers,269301,https://www.russ-davies.com/alter-egos
854,855,Ex:Re,58725,https://shop.4ad.com/format/1134848-exre
946,947,Hallucinogen,37698,https://hallucinogenmusic.bandcamp.com/album/t...


MusicBrainz matching-person leads: 10
Biographies available / keyword leads: 970 67
Group records also linked to individual members: 4


## Origin and FUA coverage are different quantities

An origin label exists for 749 bands, but a strict Functional Urban Area assignment exists for only 663. A further three enter the extended mapping. Broad country/nation labels are counted as resolved by the catalogue, so label completeness does not establish precise formation locality. The audit's proposals and source variations remain pending editorial evidence.

In [4]:
ledger = ledger.merge(origins[['returned_spotify_id','final_status','final_origin','manual_status','review_priority']], on='returned_spotify_id', validate='one_to_one')
ledger = ledger.merge(fua[['returned_spotify_id','mapping_tier','fua_code']], on='returned_spotify_id', validate='one_to_one')
check('All origin audit IDs and values reconcile', origins.returned_spotify_id.tolist()==c.returned_spotify_id.tolist() and origins.monthly_listeners.tolist()==c.monthly_listeners.tolist())
rebuilt_origin = build_audit(musicbrainz, wikipedia)
compare('Origin audit rebuild', rebuilt_origin[['final_status','final_origin','current_claim_place']], origins[['final_status','final_origin','current_claim_place']])
check('Origin status report matches saved audit', origins.final_status.value_counts().to_dict()==origin_report['final_status_counts'])
decided = origins.final_status.isin(['corrected','resolved'])
check('All recorded corrections and resolutions reflected in current origin clusters', origins.loc[decided,'final_origin'].tolist()==c.loc[decided,'origin_cluster'].tolist())
origin_status = origins.groupby('final_status', as_index=False).agg(bands=('spotify_name','size'), monthly_listeners=('monthly_listeners','sum'))
origin_status['band_share'] = origin_status.bands/1000
origin_status['listener_share'] = origin_status.monthly_listeners/total
deciles = c.assign(rank_band=((c.popularity_rank-1)//100+1), resolved=c.origin_cluster.ne(''), strict=fua.mapping_tier.eq('strict')).groupby('rank_band', as_index=False).agg(bands=('spotify_name','size'), origins=('resolved','sum'), strict_fua=('strict','sum'), monthly_listeners=('monthly_listeners','sum'))
deciles['rank_range'] = deciles.rank_band.map(lambda x:f'{(x-1)*100+1}–{x*100}')
deciles['origin_share'] = deciles.origins/deciles.bands
deciles['strict_fua_share'] = deciles.strict_fua/deciles.bands
broad = c[c.origin_cluster.isin(['Scotland','Wales','Northern Ireland','United States'])]
coverage_rows=[]
for label,mask in [('Origin label present',c.origin_cluster.ne('')),('Strict FUA',fua.mapping_tier.eq('strict')),('Extended FUA',fua.mapping_tier.isin(['strict','reviewed_extended']))]:
    coverage_rows.append({'coverage':label,'bands':int(mask.sum()),'band_share':float(mask.mean()),'listener_share':float(c.loc[mask,'monthly_listeners'].sum()/total),'follower_share':float(c.loc[mask,'followers'].sum()/followers_total)})
display(pd.DataFrame(coverage_rows)); display(origin_status); display(deciles)
pending = origins[origins.final_status.isin(['evidence_for_missing_claim','source_variation','unresolved','contested'])]
print('Pending or contested:',len(pending),'listener share',pending.monthly_listeners.sum()/total)

,coverage,bands,band_share,listener_share,follower_share
0,Origin label present,749,0.749,0.947937,0.962647
1,Strict FUA,663,0.663,0.907385,0.923456
2,Extended FUA,666,0.666,0.909593,0.925666


,final_status,bands,monthly_listeners,band_share,listener_share
0,confirmed,641,2061045048,0.641,0.894442
1,contested,2,7293135,0.002,0.003165
2,corrected,18,56284557,0.018,0.024426
3,evidence_for_missing_claim,187,91778117,0.187,0.039829
4,resolved,11,11209685,0.011,0.004865
5,source_variation,68,33520419,0.068,0.014547
6,unresolved,73,43149540,0.073,0.018726


,rank_band,bands,origins,strict_fua,monthly_listeners,rank_range,origin_share,strict_fua_share
0,1,100,100,97,1636569832,1–100,1.00,0.97
1,2,100,86,80,358773600,101–200,0.86,0.80
2,3,100,84,77,143670036,201–300,0.84,0.77
3,4,100,78,69,67962154,301–400,0.78,0.69
4,5,100,75,61,37975404,401–500,0.75,0.61
5,6,100,69,57,24586884,501–600,0.69,0.57
6,7,100,72,60,15642077,601–700,0.72,0.60
7,8,100,67,63,9348695,701–800,0.67,0.63
8,9,100,57,48,5985624,801–900,0.57,0.48
9,10,100,61,51,3766195,901–1000,0.61,0.51


Pending or contested: 330 listener share 0.07626728209683357


## Population totals and downstream outputs reproduce

The denominator is the frozen set of 83 OECD FUAs with 2024 population data. The 24 zero-count FUAs have no mapped selected acts; this does not establish that they have no bands. Of 59 represented FUAs, 25 have only one selected band. Model intervals describe conditional sampling assumptions and do not include systematic origin or eligibility error.

In [5]:
attached = attach_fua_population(c, mapping, population)
check('Population keys, year and positive denominators', population.fua_code.is_unique and population.population.gt(0).all() and population.population_year.eq(2024).all())
check('FUA audit has exact catalogue ID order and measures', fua.returned_spotify_id.tolist()==c.returned_spotify_id.tolist() and fua.monthly_listeners.tolist()==c.monthly_listeners.tolist())
compare('FUA audit assignments from reference mapping', attached[['returned_spotify_id','fua_code','mapping_tier']], fua[['returned_spotify_id','fua_code','mapping_tier']])
for view,tiers in [('strict',{'strict'}),('extended',{'strict','reviewed_extended'})]:
    result, _ = build_population_adjusted_metrics(attached, included_tiers=tiers)
    compare(f'{view.title()} FUA aggregate rebuild', result, read(f'data/processed/{base}_population_{view}.csv'))
shares, share_coverage = build_output_share_metrics(c, fua, population, included_tiers={'strict','reviewed_extended'})
compare('83-FUA output share rebuild', shares, read('artifacts/experiments/top1000_output_share_vs_population/20260718T204522Z/fua_output_shares_extended.csv'))
models = {}
for name, fit in [('negative_binomial',fit_negative_binomial_band_scaling),('loglog_follower',fit_loglog_follower_scaling)]:
    result, summary = fit(shares)
    saved = read(f'artifacts/experiments/top1000_scaling_models/20260718T204522Z/{name}_fua_results.csv')
    compare(f'{name} model output rebuild', result, saved)
    models[name]=summary
d = pd.DataFrame(dashboard['bands']).sort_values('catalogRank').reset_index(drop=True)
for dc,cc in {'id':'returned_spotify_id','name':'spotify_name','catalogRank':'popularity_rank','monthlyListeners':'monthly_listeners','followers':'followers','originCluster':'origin_cluster'}.items():
    check(f'Dashboard {dc} reconciles', d[dc].fillna('').tolist()==c[cc].tolist())
check('Dashboard uses strict FUA assignments', d.fuaCode.fillna('').tolist()==fua.fua_code.where(fua.mapping_tier.eq('strict'),'').tolist())
for area in dashboard['fuas']:
    rows = d[d.fuaCode.eq(area['id'])]
    check(f'Dashboard aggregate {area["label"]}', int(rows.monthlyListeners.sum())==area['monthlyListenersTotal'] and int(rows.followers.sum())==area['followersTotal'] and len(rows)==area['bandCount'])
display(pd.Series(share_coverage))

selected_bands                   1.000000e+03
mapped_bands                     6.660000e+02
mapped_band_share                6.660000e-01
selected_monthly_listeners       2.304281e+09
mapped_monthly_listener_share    9.095926e-01
selected_followers               9.277915e+08
mapped_follower_share            9.256664e-01
population_fuas                  8.300000e+01
mapped_fuas                      5.900000e+01
zero_band_fuas                   2.400000e+01
population_total                 5.389698e+07
dtype: float64

## Enrichment and freshness require separate caveats

Genre and inception-year fields are Wikidata-derived features, not a manual history panel. Wikipedia pageviews measure a different audience and have partial or missing capture coverage. A frozen July snapshot remains reproducible but must not be labelled current September reach. Exact capture-query text and per-response measurement times are not preserved in the source response.

In [6]:
genre = read('artifacts/experiments/genre_city_histories/20260725/band_genre_year_audit.csv')
pageviews = read('artifacts/experiments/beyond_spotify/20250701_20260630/band_pageview_audit.csv')
for label,frame in [('Genre/year',genre),('Wikipedia',pageviews)]:
    check(f'{label} has all 1,000 unique catalogue IDs', frame.returned_spotify_id.is_unique and set(frame.returned_spotify_id)==set(c.returned_spotify_id))
    aligned=frame.set_index('returned_spotify_id').loc[c.returned_spotify_id]
    check(f'{label} frozen audience values match', aligned.monthly_listeners.tolist()==c.monthly_listeners.tolist() and aligned.followers.tolist()==c.followers.tolist())
partial = pageviews[pageviews.pageview_months.between(1,11)]
enrichment = {'with_genre':int(genre.genre_labels.ne('').sum()),'with_year':int(genre.inception_year.ne('').sum()),'pageview_status':pageviews.pageview_status.value_counts().to_dict(),'partial_pageview_rows':partial[['spotify_name','pageview_months','pageviews_total']].to_dict('records')}
filtered_first = identity[identity.identity_accepted & identity.band_eligible].drop_duplicates('returned_spotify_id').drop_duplicates('wikidata_qid')
suppressed = filtered_first[~filtered_first.capture_key.isin(pool.capture_key)]
check('Deduplication ordering does not change this top 1,000 cutoff', suppressed.monthly_listeners.lt(cutoff).all())
concentration = {str(n):float(c.head(n).monthly_listeners.sum()/total) for n in [1,10,20,100,200]}
solo_listeners=int(solo.monthly_listeners.sum())
replacement=pool.iloc[1000:1005]
summary = {
 'review_date':'2026-09-05','assessment':'Needs revision for bands-only and city-completeness claims',
 'rows':len(c),'columns':len(c.columns),'total_monthly_listeners':total,'total_followers':followers_total,
 'capture_timestamp':c.stats_extracted_at_utc.unique().tolist(),'funnel':funnel,
 'missing_fields':c.eq('').sum().loc[lambda x:x>0].to_dict(),'coverage':coverage_rows,
 'origin_status':origin_status.to_dict('records'),'origin_deciles':deciles.to_dict('records'),
 'verified_solo':solo[['popularity_rank','spotify_name','monthly_listeners','eligibility_source_url']].to_dict('records'),
 'solo_listener_share':solo_listeners/total,'solo_listener_total':solo_listeners,
 'five_removal_sensitivity':{'new_cutoff':int(replacement.monthly_listeners.min()),'added_names':replacement.spotify_name.tolist(),'net_listener_change':int(replacement.monthly_listeners.sum())-solo_listeners},
 'matching_person_leads':ledger[ledger.matching_musicbrainz_person.ne('')][['popularity_rank','spotify_name','monthly_listeners']].to_dict('records'),
 'member_link_rows':ledger[ledger.other_person_ids_in_musicbrainz_links.ne('')][['spotify_name','other_person_ids_in_musicbrainz_links']].to_dict('records'),
 'bios_available':sum(bio_present),'bio_keyword_leads':sum(bio_flags),'broad_origin_labels':broad[['spotify_name','origin_cluster']].to_dict('records'),
 'pending_or_contested_rows':len(pending),'pending_listener_share':float(pending.monthly_listeners.sum()/total),
 'share_coverage':share_coverage,'one_band_fuas':int(shares.band_count.eq(1).sum()),'models':models,
 'listener_concentration':concentration,'enrichment':enrichment,
 'dedup_order_suppressed_below_cutoff':suppressed[['spotify_name','monthly_listeners']].to_dict('records'),
 'origin_corrections_and_resolutions_applied':int(decided.sum()),
 'checks':checks,'source_sha256':sources,
}
ledger.to_csv(OUT / 'row_review.csv', index=False)
(OUT / 'evidence.json').write_text(json.dumps(summary, indent=2, default=lambda x:x.item() if hasattr(x,'item') else str(x))+'\n')
display(pd.Series(enrichment)); print(f'{len(checks)} assertions/reconciliations passed; all 1,000 rows saved to row_review.csv.')

with_genre                                                             928
with_year                                                              828
pageview_status          {'ok': 924, 'not_captured': 37, 'no_enwiki_art...
partial_pageview_rows    [{'spotify_name': 'Death Of Guitar Pop', 'page...
dtype: object

99 assertions/reconciliations passed; all 1,000 rows saved to row_review.csv.


## Required next steps and limitations

1. Adjudicate the five verified solo-project exclusions and the remaining matching-person cases, using the existing override table. Decide whether former groups/current solo acts are in scope. Backfill from the ranked eligible pool only after reviewing replacement identities.
2. Prioritize pending origins by both listener weight and their impact on small FUA counts. Do not promote proposals automatically: Studio Killers currently gets a country, Finland, from a multi-country source, not a formation locality.
3. **Completed 5 September 2026:** the origin and quality audit notebooks now state the current counts, the pending eligibility findings and the distinction between origin-label and FUA coverage. The 29 origin corrections/resolutions were already applied; this follow-up updated prose only.
4. Make MusicBrainz evidence identity-aware. Four group records also link to eight individual members. Their extra records currently add no distinct begin-area, but must not establish group formation evidence.
5. Present FUA coverage and one-band dependence beside all city comparisons; retain the zero-FUA universe and distinguish observed zero selected acts from zero musical activity.
6. Keep the exact snapshot date, candidate-frame limitation, incomplete pageview windows and historical/current metric distinction visible. On the next capture preserve exact query text/hash, request timing and input hashes.

The machine checks cover every row; primary-source eligibility adjudication is targeted. They do not prove that every Spotify page belongs to the intended artist or that every formation claim is true. The review does not establish the candidate frame's exhaustiveness. No live Spotify values were collected and no source datasets were edited.